# S6E5 — Tuning dos Modelos

## 1. Imports e configuração

In [ ]:
from pathlib import Path

import optuna

from mltemplate.config import ProjectConfig
from mltemplate.storage import StorageManager
from mltemplate.data import KaggleSource, DataManager
from mltemplate.tuning import OptunaTuner, XGBoostAdapter, LightGBMAdapter

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
config = ProjectConfig(
    target="PitNextLap",
    numerical_features=[],    # manter igual ao 02_features.ipynb
    categorical_features=[],  # manter igual ao 02_features.ipynb
    ignore_features=["id"],
    problem_type="regression",
)

storage = StorageManager(root=Path("."))
dm      = DataManager(storage, config)

## 2. Carregar feature set e targets

In [ ]:
X_train_fe, X_test_fe = dm.load_feature_set(name="v1")

source = KaggleSource("playground-series-s6e5")
train_df, _ = dm.load_raw(source)
_, _, y_train, y_val = dm.split(train_df)

print(f"X_train: {X_train_fe.shape}")
print(f"X_test:  {X_test_fe.shape}")

## 3. Tuning XGBoost

In [ ]:
def xgb_param_space(trial: optuna.Trial) -> dict:
    return {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }

result_xgb = OptunaTuner(config).tune(
    XGBoostAdapter(),
    X_train_fe, y_train,
    param_space_func=xgb_param_space,
    scoring="rmse",
    n_trials=50,
)

print(f"XGB — RMSE: {result_xgb.score:.4f}")
print(f"Params: {result_xgb.params}")

In [ ]:
storage.save_model(result_xgb.model, "xgb_v1")
storage.save_metrics({"rmse": result_xgb.score, "params": result_xgb.params}, "xgb_v1")
print("Modelo e métricas XGB salvos.")

## 4. Tuning LightGBM

In [ ]:
def lgbm_param_space(trial: optuna.Trial) -> dict:
    return {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 1000),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 300),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

result_lgbm = OptunaTuner(config).tune(
    LightGBMAdapter(),
    X_train_fe, y_train,
    param_space_func=lgbm_param_space,
    scoring="rmse",
    n_trials=50,
)

print(f"LGBM — RMSE: {result_lgbm.score:.4f}")
print(f"Params: {result_lgbm.params}")

In [ ]:
storage.save_model(result_lgbm.model, "lgbm_v1")
storage.save_metrics({"rmse": result_lgbm.score, "params": result_lgbm.params}, "lgbm_v1")
print("Modelo e métricas LGBM salvos.")

## 5. [Opcional] Submissão direta do melhor modelo

In [ ]:
import pandas as pd

# best_model = result_xgb.model   # ou result_lgbm.model
# name       = "xgb_v1"           # ajustar conforme o modelo escolhido

# _, test_df = dm.load_raw(KaggleSource("playground-series-s6e5"))
# preds      = best_model.predict(X_test_fe)

# submission = pd.DataFrame({"id": test_df["id"], config.target: preds})
# storage.save_submission(submission, name)
# print(f"Submissão salva em: {storage.submissions_path / (name + '.csv')}")
# submission.head()